[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hoax12/fable-pytorch/blob/main/notebooks/15_beyond_the_forge_onnx.ipynb)

# ⚒️ Epilogue · Quest 15 — Beyond the Forge — ONNX

*Your model must now survive OUTSIDE the forge: no PyTorch, no Python class, just a portable artifact.*

⬅️ [14_final_boss](14_final_boss.ipynb)

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math, random

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')
np.random.seed(0); random.seed(0)

# ══════════════════ ⚒️ FORGE GRADER — run me once ══════════════════
# Powers check() and xp_report(). Exercises give instant feedback + XP.
_XP = {"earned": 0, "done": set(), "checks": {}}

def _register(name, points, test, hint):
    _XP["checks"][name] = (points, test, hint)

def check(name, *answer):
    """Grade an exercise: check("ex1", your_answer). Repeatable until correct."""
    if name not in _XP["checks"]:
        print(f"unknown exercise: {name!r} — available: {list(_XP['checks'])}")
        return
    points, test, hint = _XP["checks"][name]
    try:
        ok = bool(test(*answer))
        err = None
    except Exception as e:
        ok, err = False, f"{type(e).__name__}: {e}"
    if ok:
        first = name not in _XP["done"]
        if first:
            _XP["done"].add(name)
            _XP["earned"] += points
        total = sum(p for p, _, _ in _XP["checks"].values())
        tag = f"+{points} XP" if first else "already earned"
        print(f"✅ {name} — correct! {tag}   ⚡ {_XP['earned']}/{total} XP")
    else:
        msg = f"❌ {name} — not yet."
        if err:
            msg += f" (your answer raised {err})"
        print(msg + f"\n   💡 hint: {hint}")

def xp_report():
    total = sum(p for p, _, _ in _XP["checks"].values())
    n = len(_XP["checks"])
    bar = "█" * int(10 * _XP["earned"] / max(total, 1)) or "░"
    print(f"⚡ XP: {_XP['earned']}/{total}   [{bar:<10}]   exercises: {len(_XP['done'])}/{n}")
    for name in _XP["checks"]:
        print(("  ✅ " if name in _XP["done"] else "  ⬜ ") + name)

# onnx + onnxruntime: auto-install on Colab if missing
import sys, subprocess, importlib
for pkg in ["onnx", "onnxruntime"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import onnx
import onnxruntime as ort
import os, time, json
print(f"onnx {onnx.__version__} | onnxruntime {ort.__version__}")

## The last door

Everything you've trained so far lives *inside* PyTorch — a Python object that needs your class
definition and the whole torch runtime to make a single prediction. Production usually can't
afford that: a C++ game server, a phone, a browser tab, a tiny container.

**ONNX** (Open Neural Network Exchange) is the passport out:

```
PyTorch ──torch.onnx.export──▶ model.onnx ──▶ ONNX Runtime · TensorRT · CoreML · the browser
```

**ONNX Runtime** executes `.onnx` files anywhere, usually *faster* than eager PyTorch on CPU —
and it needs only `onnxruntime` + `numpy` at serving time.

### The quest
1. 🏗️ Train a glyph classifier (your last one — savor it)
2. 📦 Export to ONNX with a **dynamic batch axis**
3. 🔍 Open the artifact and inspect the graph inside
4. ✅ Serve with ONNX Runtime; prove it matches PyTorch
5. ⏱️ Race them: eager vs ONNX Runtime
6. 🪶 Quantize the artifact to int8
7. 🚀 A torch-free oracle (`ship/glyph_oracle.py`)

## 1 · 🏗️ Train the traveler

In [ ]:
# --- The Glyph dataset: ✕ ◯ ┼ ╱  (self-contained, no torchvision needed) ----
GLYPHS = ["cross", "ring", "plus", "slash"]

def _draw_glyph(cls, size=20, rng=None):
    rng = rng or torch.Generator().manual_seed(0)
    img = torch.zeros(size, size)
    ys, xs = torch.meshgrid(torch.arange(size), torch.arange(size), indexing="ij")
    jx = int(torch.randint(-2, 3, (1,), generator=rng))   # random jitter
    jy = int(torch.randint(-2, 3, (1,), generator=rng))
    c = size // 2
    if cls == 0:    # cross ✕ : two diagonals
        img[((xs - ys).abs() + (jx - jy)).abs() <= 1] = 1.0
        img[((xs + ys - size + 1) + jx + jy).abs() <= 1] = 1.0
    elif cls == 1:  # ring ◯
        r2 = (xs - c - jx) ** 2 + (ys - c - jy) ** 2
        img[(r2 >= 25) & (r2 <= 49)] = 1.0
    elif cls == 2:  # plus ┼
        img[(ys - c - jy).abs() <= 1] = 1.0
        img[(xs - c - jx).abs() <= 1] = 1.0
    else:           # slash ╱ : one diagonal
        img[((xs + ys - size + 1) + jx + jy).abs() <= 1] = 1.0
    img = img + 0.08 * torch.randn(size, size, generator=rng)
    return img.clamp(0, 1)

def make_glyphs(n_per_class=300, size=20, seed=0):
    rng = torch.Generator().manual_seed(seed)
    X = torch.stack([_draw_glyph(c, size, rng) for c in range(4) for _ in range(n_per_class)])
    y = torch.arange(4).repeat_interleave(n_per_class)
    perm = torch.randperm(len(y), generator=rng)
    return X[perm].unsqueeze(1), y[perm]   # (N, 1, 20, 20), (N,)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

X, y = make_glyphs(n_per_class=300)
train_loader = DataLoader(TensorDataset(X[:960], y[:960]), batch_size=64, shuffle=True)
X_test, y_test = X[960:], y[960:]

traveler = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 20 -> 10
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 10 -> 5
    nn.Flatten(), nn.Linear(32 * 5 * 5, 64), nn.ReLU(), nn.Linear(64, 4),
).to(device)
opt = torch.optim.Adam(traveler.parameters(), lr=1.5e-3)
for epoch in range(4):
    traveler.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); F.cross_entropy(traveler(xb), yb).backward(); opt.step()

traveler.eval()
with torch.no_grad():
    acc = (traveler(X_test.to(device)).argmax(1).cpu() == y_test).float().mean()
print(f"traveler trained — test accuracy {acc*100:.1f}%")

## 2 · 📦 Export

`torch.onnx.export` traces the model once with an example input, records the graph, and writes
it out. Two details matter:

- **`dynamic_shapes=({0: "batch"},)`** marks dim 0 as symbolic — without it, the artifact only
  accepts the exact batch size you exported with. (Older code uses `dynamic_axes=`; same idea.)
- **`external_data=False`** embeds the weights so you get ONE portable file instead of
  `glyphs.onnx` + a `glyphs.onnx.data` sidecar.

In [ ]:
os.makedirs("ship", exist_ok=True)
PATH = "ship/glyphs.onnx"

traveler.cpu().eval()
torch.onnx.export(
    traveler, (torch.randn(1, 1, 20, 20),), PATH,
    input_names=["glyph"], output_names=["logits"],
    dynamic_shapes=({0: "batch"},),      # dim 0 of the (only) input is symbolic
    opset_version=18,
    external_data=False,                 # one self-contained file
)
with open("ship/glyph_classes.json", "w") as f:
    json.dump(GLYPHS, f)
print(f"\nexported {PATH} ({os.path.getsize(PATH) / 1e3:.1f} KB) + class names")

## 3 · 🔍 What's inside the artifact?

In [ ]:
model_proto = onnx.load(PATH)
onnx.checker.check_model(model_proto)      # raises if malformed
print("graph valid ✅\n")

def dims(t):
    return [d.dim_param or d.dim_value for d in t.type.tensor_type.shape.dim]

print("input :", [(i.name, dims(i)) for i in model_proto.graph.input])
print("output:", [(o.name, dims(o)) for o in model_proto.graph.output])
print("ops   :", sorted({n.op_type for n in model_proto.graph.node}))

The batch dimension reads `"batch"` — a *symbol*, not a number. And your `nn.Sequential` has
become a flat list of standard ops (`Conv`, `Relu`, `MaxPool`, `Gemm`...) any runtime can execute.
No Python. No class. No torch.

## 4 · ✅ Serve it — and prove nothing was lost in translation

In [ ]:
sess = ort.InferenceSession(PATH, providers=["CPUExecutionProvider"])

x_np = X_test.numpy()
with torch.no_grad():
    torch_logits = traveler(X_test).numpy()
ort_logits = sess.run(["logits"], {"glyph": x_np})[0]

diff = np.abs(torch_logits - ort_logits).max()
print(f"max |pytorch − onnxruntime| = {diff:.2e}  -> match: {diff < 1e-4} ✅")

# dynamic batch in action: same artifact, any batch size
for b in (1, 7, 64):
    out = sess.run(["logits"], {"glyph": np.random.rand(b, 1, 20, 20).astype(np.float32)})[0]
    print(f"batch {b:2d} -> logits {out.shape}")

## 5 · ⏱️ The race — eager PyTorch vs ONNX Runtime (CPU)

In [ ]:
big = np.random.rand(128, 1, 20, 20).astype(np.float32)
big_t = torch.from_numpy(big)

def clock(fn, reps=60):
    fn()
    t0 = time.time()
    for _ in range(reps):
        fn()
    return (time.time() - t0) / reps * 1000

t_torch = clock(lambda: traveler(big_t))
t_ort = clock(lambda: sess.run(["logits"], {"glyph": big}))
print(f"PyTorch eager: {t_torch:6.2f} ms/batch")
print(f"ONNX Runtime : {t_ort:6.2f} ms/batch   ({t_torch / t_ort:.2f}x)")

## 6 · 🪶 Quantize the artifact

Dynamic int8 quantization converts weights to 8-bit integers, directly on the `.onnx` file —
no PyTorch involved. One real-world wrinkle: the exporter bakes cached activation shapes
(`value_info`) into the graph, and they confuse the quantizer's shape inference when the batch
axis is symbolic. We strip them first. (Welcome to deployment: half the job is little
compatibility moves like this.)

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

def quantize_onnx(src, dst):
    """Strip stale value_info (breaks shape inference with dynamic axes), then quantize."""
    m = onnx.load(src)
    del m.graph.value_info[:]
    tmp = src + ".clean"
    onnx.save(m, tmp)
    quantize_dynamic(tmp, dst, weight_type=QuantType.QInt8)
    os.remove(tmp)

Q8 = "ship/glyphs_int8.onnx"
quantize_onnx(PATH, Q8)

kb, kb8 = os.path.getsize(PATH) / 1e3, os.path.getsize(Q8) / 1e3
print(f"fp32 {kb:.1f} KB -> int8 {kb8:.1f} KB   ({kb / kb8:.1f}x smaller)")

qsess = ort.InferenceSession(Q8, providers=["CPUExecutionProvider"])
q_acc = (qsess.run(["logits"], {"glyph": x_np})[0].argmax(1) == y_test.numpy()).mean()
print(f"int8 accuracy: {q_acc*100:.1f}%  (fp32 was {acc*100:.1f}%)")

Roughly 4x smaller (float32 → int8), accuracy essentially untouched. For our toy model the KBs
don't matter — for a 7B-parameter LLM, this same idea is the difference between "fits on your
GPU" and "doesn't".

## 7 · 🚀 The oracle — inference with zero torch

This is the whole point. The class below imports **only numpy + onnxruntime**. Drop it and the
two `ship/` files on any machine and you have a prediction service. A standalone CLI version
lives at [`ship/glyph_oracle.py`](../ship/glyph_oracle.py) — run it from `fable_folder/`:

```bash
python ship/glyph_oracle.py notebooks/ship/glyphs.onnx --classes notebooks/ship/glyph_classes.json --glyph ring
```

In [ ]:
class GlyphOracle:
    """Torch-free serving: numpy in, prediction out."""
    def __init__(self, model_path, classes_path):
        self.sess = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
        self.inp = self.sess.get_inputs()[0].name
        with open(classes_path) as f:
            self.classes = json.load(f)

    def __call__(self, image_20x20):
        x = np.asarray(image_20x20, dtype=np.float32).reshape(1, 1, 20, 20)
        logits = self.sess.run(None, {self.inp: x})[0][0]
        p = np.exp(logits - logits.max()); p /= p.sum()
        k = int(p.argmax())
        return {"glyph": self.classes[k], "confidence": float(p[k])}

oracle = GlyphOracle("ship/glyphs.onnx", "ship/glyph_classes.json")
sample_idx = int((y_test == 1).nonzero()[0])       # grab a ring ◯ from the test set
print("oracle says:", oracle(X_test[sample_idx, 0].numpy()),
      "| truth:", GLYPHS[y_test[sample_idx]])

In [ ]:
# ── this notebook's exercises (run once) ───────────────────────────────
_register("warmup", 5,
    lambda s: "torch" in s.lower() or "pytorch" in s.lower() or "python" in s.lower(),
    "what dependency does an .onnx artifact free your serving code from?")
_register("dyn_batch", 20,
    lambda s: (lambda n: s.run(None, {n: np.zeros((1, 1, 20, 20), np.float32)})[0].shape[0] == 1
               and s.run(None, {n: np.zeros((5, 1, 20, 20), np.float32)})[0].shape[0] == 5)(s.get_inputs()[0].name),
    "export ANY glyph model with a dynamic dim-0 (dynamic_shapes / dynamic_axes) and submit an ort.InferenceSession for it")
_register("faithful", 15,
    lambda d: float(d) < 1e-4,
    "submit the max abs difference between pytorch and onnxruntime logits on the test set")
_register("quant_shrunk", 15,
    lambda sizes: len(sizes) == 2 and sizes[1] < sizes[0],
    "submit (fp32_kb, int8_kb) — int8 must be smaller. Remember the lesson: use a big-enough model (the chunky one qualifies)")

In [ ]:
check("warmup", "torch")

## 🏆 Boss Challenges

Earn your XP — write each answer in a cell below and grade it with `check(...)`. When you're done, run `xp_report()`.

- `dyn_batch` (20 XP) — export a model with a dynamic batch axis; submit the `InferenceSession`.
- `faithful` (15 XP) — submit the max PyTorch↔ORT logit difference as a float.
- `quant_shrunk` (15 XP) — submit `(fp32_kb, int8_kb)` after quantizing.

In [ ]:
# ⚔️ your attempts here...

# xp_report()

---
## 🌅 Epilogue complete

Your model walked out of the forge and works in a world that has never heard of PyTorch. That's
the full arc: **idea → engine → framework → mastery → artifact.**

Ideas for your own adventures: serve the oracle from FastAPI, run the `.onnx` in the browser
with ONNX Runtime Web, or export your Final Boss model and quantize it. ⚒️